In [1]:
import os
import json
import joblib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report, accuracy_score, precision_recall_fscore_support, confusion_matrix


os.makedirs('evaluation', exist_ok=True)
os.makedirs('artifacts', exist_ok=True)


print("Loading cleaned dataset...")
df = pd.read_csv('experiments/data/cleaned_train.csv')
df = df.dropna(subset=['cleaned_text'])

X = df['cleaned_text']
y = df['is_toxic']


X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)


print("Vectorizing text using TF-IDF...")
vectorizer = TfidfVectorizer(max_features=10000, ngram_range=(1, 2))
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)


models = {
    "Multinomial Naive Bayes": MultinomialNB(),
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "Linear SVM": LinearSVC(random_state=42, dual=False)
}

results = {}
best_model_name = None
best_f1 = 0.0
best_model_obj = None

print("\nTraining and evaluating models...\n")

for name, model in models.items():
    model.fit(X_train_tfidf, y_train)
    y_pred = model.predict(X_test_tfidf)
    
    acc = accuracy_score(y_test, y_pred)
    prec, rec, f1, _ = precision_recall_fscore_support(y_test, y_pred, average='binary')
    
    results[name] = {
        "accuracy": round(acc, 4),
        "precision": round(prec, 4),
        "recall": round(rec, 4),
        "f1_score": round(f1, 4)
    }
    
    print(f"=== {name} ===")
    print(f"Accuracy : {acc:.4f} | Precision: {prec:.4f} | Recall: {rec:.4f} | F1-Score: {f1:.4f}\n")
    
    if f1 > best_f1:
        best_f1 = f1
        best_model_name = name
        best_model_obj = model


with open('evaluation/metrics.json', 'w') as f:
    json.dump(results, f, indent=4)

print(f"🏆 Best Performing Model: {best_model_name} (F1-Score: {best_f1:.4f})")


best_pred = best_model_obj.predict(X_test_tfidf)

report = classification_report(y_test, best_pred, target_names=['Clean', 'Toxic'])
with open('evaluation/classification_report.txt', 'w') as f:
    f.write(f"Best Model: {best_model_name}\n\n")
    f.write(report)

cm = confusion_matrix(y_test, best_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Clean', 'Toxic'], yticklabels=['Clean', 'Toxic'])
plt.title(f'Confusion Matrix - {best_model_name}')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.tight_layout()
plt.savefig('evaluation/confusion_matrix.png', dpi=300)
plt.close()


joblib.dump(best_model_obj, 'artifacts/toxic_model.pkl')
joblib.dump(vectorizer, 'artifacts/tfidf_vectorizer.pkl')

print("\n Success! Model, Vectorizer, and Evaluation Assets saved successfully!")

Matplotlib is building the font cache; this may take a moment.


Loading cleaned dataset...
Vectorizing text using TF-IDF...

Training and evaluating models...

=== Multinomial Naive Bayes ===
Accuracy : 0.8983 | Precision: 0.8980 | Recall: 0.7840 | F1-Score: 0.8371

=== Logistic Regression ===
Accuracy : 0.9066 | Precision: 0.9297 | Recall: 0.7787 | F1-Score: 0.8476

=== Linear SVM ===
Accuracy : 0.9150 | Precision: 0.9060 | Recall: 0.8314 | F1-Score: 0.8671

🏆 Best Performing Model: Linear SVM (F1-Score: 0.8671)

 Success! Model, Vectorizer, and Evaluation Assets saved successfully!
